# Module 9.4: Graduation — Reading a Real LLM

You've built every part of a modern language model. The final exam isn't writing more code — it's **reading someone else's**. The claim of this whole course is that you can now open the source of a production model — Llama, Mistral, GPT-2 — and *recognize every component*. This module is where we cash that claim in.

Three steps:
1. Re-read **our own** library with fresh eyes (it was deliberately written in the style of Meta's Llama reference code).
2. Build a **Rosetta Stone**: our names ↔ Llama's names ↔ GPT-2's names.
3. Prove you understand the architecture by **computing Llama-3-8B's parameter count by hand** — and getting 8B.

## 1. Re-read Your Own Library

Throughout the course, notebooks re-implemented each concept inline, and `src/llm_workout/` kept the polished version. Print two of its classes and notice: at this point, every line should read as *"oh, that's just Module X"*.

In [1]:
import inspect
from llm_workout import layers, model

print(inspect.getsource(layers.RMSNorm))       # Module 4.1
print(inspect.getsource(layers.SwiGLU))        # Module 4.1

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d_model: int, hidden_dim: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
        self.w3 = nn.Linear(d_model, hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))



In [2]:
# The attention class is longer, but scan it and name each numbered step yourself:
# projections (3.1), RoPE (2.2), KV cache (7.1), scores+mask+softmax (3.1), re-assembly (3.1).
print(inspect.getsource(layers.MultiHeadCausalAttention))

class MultiHeadCausalAttention(nn.Module):
    """
    Standard Multi-Head Attention, modernized with RoPE support and KV Caching.
    """
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(
        self, 
        x: torch.Tensor, 
        mask: Optional[torch.Tensor] = None, 
        freqs_cis: Optional[torch.Tensor] = None,
        kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None
    ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        
        batch_size, seq_len, d_model = x.size()
        
        # 1. Linear pro

## 2. The Rosetta Stone

Real codebases use different *names* for the same machinery you built. This table is most of what you need to read them fluently:

| Ours (`src/llm_workout/`) | Llama-3 (HF `modeling_llama.py`) | GPT-2 (`nanoGPT` / HF) | Where you built it |
|---|---|---|---|
| `RMSNorm` | `LlamaRMSNorm` | `LayerNorm` (with mean-centering + bias) | Module 4.1 |
| `SwiGLU` (`w1`=gate, `w3`=up, `w2`=down) | `LlamaMLP` (`gate_proj`, `up_proj`, `down_proj`) | `MLP` with GELU (`c_fc`, `c_proj`) | Modules 4.1 |
| `MultiHeadCausalAttention` (`W_q/W_k/W_v/W_o`) | `LlamaAttention` (`q_proj/k_proj/v_proj/o_proj`) | `CausalSelfAttention` (one fused `c_attn`) | Module 3.1 |
| `precompute_freqs_cis` + `apply_rotary_emb` | `LlamaRotaryEmbedding` + `apply_rotary_pos_emb` | learned position table `wpe` | Module 2.2 |
| `DecoderBlock` (pre-norm, 2 residuals) | `LlamaDecoderLayer` | `Block` | Modules 4.1–4.2 |
| `GPT` (embed → blocks → norm → head) | `LlamaForCausalLM` | `GPT` | Module 4.3 |
| `generate` + `kv_caches` | `generate` + `past_key_values` / `Cache` | same idea | Modules 5.5, 7.1 |
| weight tying (`token_embedding` ↔ `lm_head`) | **untied** in Llama-3 (two separate matrices) | tied (like ours) | Module 4.3 |

> Fun fact: our `precompute_freqs_cis` isn't just *similar* to Meta's Llama reference implementation — it uses the **same function name**, on purpose, so that when you open their `model.py` you feel at home.

The differences you *will* see, and can now decode instantly:
- **Fused projections:** GPT-2 computes Q, K, V with a single `(d, 3d)` matrix (`c_attn`) instead of three separate ones. Same math, one matmul.
- **GQA:** Llama-3's `k_proj`/`v_proj` output fewer heads than `q_proj` — that's Grouped-Query Attention from Module 7.2, shrinking the KV cache.
- **Fast kernels:** production attention calls FlashAttention (Module 7.3) instead of materializing the score matrix.
- **Config plumbing:** everything is driven by a config object (`hidden_size`, `num_attention_heads`, `intermediate_size`...) — the same knobs as our `GPT(...)` arguments, renamed.

## 3. The Math: Count Llama-3-8B's Parameters By Hand

### The Concept
If you *really* understand an architecture, you can compute its size from its config. Let's write the formula once, from the components you built, and then aim it at the real Llama-3-8B configuration:

| Component | Parameters | Built in |
|---|---|---|
| Token embedding | $V \times d$ | 2.1 |
| Attention (per layer) | $d\,d$ (Q) $+\;2\,d\,d_{kv}$ (K,V) $+\;d\,d$ (O), where $d_{kv} = d \cdot \tfrac{\text{kv heads}}{\text{heads}}$ | 3.1, 7.2 |
| SwiGLU FFN (per layer) | $3 \times d \times d_{ff}$ | 4.1 |
| Two RMSNorms (per layer) | $2d$ | 4.1 |
| Final RMSNorm | $d$ | 4.3 |
| LM head | $V \times d$ (or **0** if tied with the embedding) | 4.3 |

RoPE adds **zero** parameters — the rotations are pure math (Module 2.2). That's one of its selling points.

In [3]:
def count_params(vocab, d_model, n_layers, n_heads, n_kv_heads, d_ff, tied):
    """Parameter count for the Llama-style decoder-only architecture we built."""
    head_dim = d_model // n_heads
    d_kv = head_dim * n_kv_heads                      # K/V output width (GQA shrinks this)
    attn = d_model * d_model + 2 * d_model * d_kv + d_model * d_model   # Q + K,V + O
    ffn = 3 * d_model * d_ff                          # SwiGLU: gate, up, down
    norms = 2 * d_model                               # attn_norm + ffn_norm
    per_layer = attn + ffn + norms
    embed = vocab * d_model
    head = 0 if tied else vocab * d_model
    return embed + n_layers * per_layer + d_model + head

# The published Llama-3-8B configuration:
llama3_8b = dict(vocab=128_256, d_model=4096, n_layers=32, n_heads=32,
                 n_kv_heads=8, d_ff=14_336, tied=False)

total = count_params(**llama3_8b)
print(f"Llama-3-8B by our formula: {total:,}  ({total/1e9:.2f}B)")
print("Meta's stated size:        ~8.0B     <- your architecture knowledge is complete.")

Llama-3-8B by our formula: 8,030,261,248  (8.03B)
Meta's stated size:        ~8.0B     <- your architecture knowledge is complete.


In [4]:
# Two follow-up questions the formula can answer instantly:

# (a) What did GQA save? Recompute with full multi-head K/V (32 kv heads, like our library):
mha = count_params(**{**llama3_8b, 'n_kv_heads': 32})
print(f"With full MHA instead of GQA: {mha/1e9:.2f}B  "
      f"(GQA saved {(mha-total)/1e6:.0f}M params -- and, far more importantly,")
print("  a 4x smaller KV cache at inference time; see Modules 7.1-7.2)")

# (b) Does the formula agree with REAL code? Aim it at our own capstone GPT:
from llm_workout.model import GPT
import torch
capstone_cfg = dict(vocab=65, d_model=128, n_layers=4, n_heads=4,
                    n_kv_heads=4, d_ff=512, tied=True)
formula = count_params(**capstone_cfg)
actual = sum(p.numel() for p in GPT(vocab_size=65, d_model=128, num_layers=4,
                                    num_heads=4, hidden_dim=512).parameters())
print(f"\nCapstone model -- formula: {formula:,}   actual: {actual:,}   match: {formula == actual}")

With full MHA instead of GQA: 8.84B  (GQA saved 805M params -- and, far more importantly,
  a 4x smaller KV cache at inference time; see Modules 7.1-7.2)

Capstone model -- formula: 1,058,048   actual: 1,058,048   match: True


## 4. What Real Models Add (That You Can Now Look Up)

Nothing below is architecturally new to you — each is a variation on a module you built:

| In the wild | What it is | Your reference point |
|---|---|---|
| `rope_theta = 500000` + RoPE scaling | stretching RoPE for 128k-token contexts | Module 2.2 |
| GQA in the shipped weights | fewer KV heads than Q heads | Module 7.2 |
| FlashAttention / SDPA kernels | tiled, memory-aware attention | Module 7.3 |
| `torch.bfloat16` everywhere | 16-bit training/inference precision | Module 9.1 (quantization is the same tradeoff, pushed further) |
| FSDP / tensor parallel | sharding one model across many GPUs | Module 8.2's compute budgets are *why*; the *how* is systems engineering |
| MoE layers (Mixtral, DeepSeek) | FFN replaced by routed experts | Module 8.1 |
| Sliding-window attention (Mistral) | causal mask limited to a local window | Module 4.2's mask, with a second constraint |
| Speculative / multi-token decoding | draft-and-verify serving | Module 9.2 |

## 5. Your Graduation Reading List

Open these — in this order — and check that you can name every component. That's the real diploma.

1. **karpathy/nanoGPT — `model.py`** (~300 lines). GPT-2 style: LayerNorm, GELU MLP, learned positions, fused `c_attn`. Everything else is your Module 4.3.
2. **meta-llama/llama3 — `llama/model.py`** (~300 lines). You will literally find `precompute_freqs_cis`, `apply_rotary_emb`, `RMSNorm`, and a `FeedForward` with `w1/w2/w3` — our library is a teaching-sized copy of this file's design.
3. **Hugging Face `transformers` — `models/llama/modeling_llama.py`**. Longer, because production: config plumbing, cache classes, attention-backend dispatch. The *architecture* inside is table 2 above.

When you can skim all three and feel bored rather than lost — congratulations. You don't just *use* LLMs anymore; you know what they *are*.

**Where to next:** Part II (notebooks 30+) switches sides — from building models to building *systems around* models: prompting, retrieval, RAG, and evaluation.

### 🏋️ Try it yourself

1. **Scale the formula.** Llama-3-**70B**'s config: `d_model=8192`, `n_layers=80`, `n_heads=64`, `n_kv_heads=8`, `d_ff=28672`, same vocab, untied. Plug it into `count_params` — do you land near 70B?
2. **Five differences.** Open nanoGPT's `model.py` and list five concrete differences from our `GPT` (think: normalization, activation, positions, projections, tying). For each, name the module of this course where the difference is explained.
3. **Upgrade the library.** Swap `GroupedQueryAttention` (your Module 7.2 implementation) into a copy of `src/llm_workout/layers.py` and re-run `pytest` — which tests still pass, and what does the KV-cache equivalence test tell you when you get it right?

In [5]:
# Starter for Task 1:
llama3_70b = dict(vocab=128_256, d_model=8192, n_layers=80, n_heads=64,
                  n_kv_heads=8, d_ff=28_672, tied=False)
total_70b = count_params(**llama3_70b)
print(f"Llama-3-70B by the formula: {total_70b/1e9:.2f}B")

Llama-3-70B by the formula: 70.55B
